# 05 - Rendering static element overlay maps

The second output product: for a chosen element, a colour-coded overlay of its relative
abundance drawn directly onto the lunar basemap and saved as a PNG.

Unlike notebook 04, this one works from the *raw* footprint geometry rather than the
regridded table. Each catalogue row is drawn as the quadrilateral it actually was, with its
four corners projected into pixel space and its outline stroked in a viridis colour keyed
to the element's share of total fitted flux. Normalisation is per element and over the
whole map, so what each figure shows is spatial contrast within that element rather than
absolute weight percent.

Oxygen is excluded from the totals. Its line sits on the artefact discussed in notebook 02
and its fitted area is the least trustworthy of the set, so including it would distort every
other element's share.

**Expected input:** the concatenated catalogue CSV from notebook 02 and an equirectangular
basemap image.

**Expected output:** one PNG per (element, opacity) pair under `results/element_overlays/`.

## Configuration

Set the element and overlay opacity here and re-run the notebook; the published set in
`results/element_overlays/` was produced this way, at 40% and 70% opacity.

In [ ]:
# On a hosted runtime:  !pip install rasterio opencv-python pillow matplotlib pandas numpy

import os
import cv2
from PIL import Image
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, get_cmap
import random

# Element to render. One of: Na, Mg, Al, Si, Ca, Ti, Mn, Fe.
ELEMENT = "Ca"

# Overlay strength. Lower values keep the underlying geography legible.
OPACITY = 0.4

CATALOGUE_CSV = os.environ.get("CATALOGUE_CSV", "../data/interim/coadded_catalogue.csv")
BASEMAP_PATH = os.environ.get("LUNAR_BASEMAP", "../data/basemap/lunar_albedo_equirect.png")
OUTPUT_DIR = os.environ.get("OVERLAY_DIR", "../results/element_overlays")

RATIO_COLUMN = f"{ELEMENT}_area_ratio"

In [ ]:
# When running on a hosted notebook backed by cloud storage, mount it first.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Development fallback for fetching the catalogue from cloud storage:
# !gdown 1sNUKLwdllYqZIGna4IiiNq-pw4hW3q6y

## Turning fitted areas into per-element shares

Each element's share is its fitted line area divided by the summed area of all elements in
that observation. Uncertainty columns play no part in a static figure and are dropped, and
rows where a fit failed contribute zero rather than propagating NaN into the colour scale.

In [ ]:
df = pd.read_csv(CATALOGUE_CSV)

# Total fitted flux per observation, oxygen excluded.
area_columns = [col for col in df.columns if "area" in col and col != "O_area"]
df["Total_area"] = df[area_columns].sum(axis=1)

# Uncertainties are irrelevant to a static overlay.
uncertainty_columns = [col for col in df.columns if "uncertainity" in col]
df.drop(columns=uncertainty_columns, inplace=True)

# Oxygen's fit is the least reliable of the set; keep it out of the totals.
df.drop(columns=["O_area"], inplace=True, errors="ignore")

# A failed fit contributes nothing rather than poisoning the colour scale.
df.fillna(0, inplace=True)

# Share of total flux attributable to each element.
for col in area_columns:
    if col != "O_area":
        df[f"{col}_ratio"] = df[col] / df["Total_area"]

# Keep only geometry and shares.
lat_lon_columns = [col for col in df.columns if "lat" in col or "lon" in col]
ratio_columns = [col for col in df.columns if "_ratio" in col]
new_df = df[lat_lon_columns + ["Total_area"] + ratio_columns]

new_df.head()

In [ ]:
# Renormalise so the plotted shares sum to one per observation.
ratio_columns = ['Na_area_ratio', 'Mg_area_ratio', 'Al_area_ratio', 'Ca_area_ratio', 'Ti_area_ratio', 'Mn_area_ratio', 'Fe_area_ratio']

# Row-wise total across the elements that are actually plotted.
row_sums = new_df[ratio_columns].sum(axis=1)


new_df[ratio_columns] = new_df[ratio_columns].div(row_sums, axis=0)
new_df.head()

## Basemap and projection constants

The overlay is drawn with OpenCV, which works in BGR, so the basemap is converted on the way
in and back again before display.

In [ ]:
lunar_map_path = BASEMAP_PATH
Image.MAX_IMAGE_PIXELS = None
lunar_map_pil = Image.open(lunar_map_path)
lunar_map = np.array(lunar_map_pil)
lunar_map = cv2.cvtColor(lunar_map, cv2.COLOR_RGB2BGR)

In [ ]:
# Full-sphere extent of the equirectangular basemap.
min_lat, max_lat = -90, 90
min_lon, max_lon = -180, 180
height, width, _ = lunar_map.shape
opacity = OPACITY

## Colour scale

Normalisation spans the observed range of the chosen element across the entire map, which
maximises visible contrast for that element. It also means colours are not comparable
between two different element figures - only spatial patterns within one figure are.

In [ ]:
# Map the element's full observed range onto the colormap.
norm = Normalize(vmin=new_df[RATIO_COLUMN].min(), vmax=new_df[RATIO_COLUMN].max())

colormap = get_cmap('viridis')
scalar_map = ScalarMappable(norm=norm, cmap=colormap)

## Drawing the footprints

Every catalogue row is stroked as its own quadrilateral. Drawing outlines rather than filled
polygons is what gives the published figures their woven texture: overlapping orbital tracks
stay individually visible, so the map doubles as a coverage plot.

In [ ]:
max_opacity=1.0
min_opacity=0.0
def lat_lon_to_pixel(lat, lon, img_width, img_height):
    """Project a coordinate onto the equirectangular basemap, in pixels."""
    x = int(((lon - min_lon) / (max_lon - min_lon)) * img_width)
    y = int(((max_lat - lat) / (max_lat - min_lat)) * img_height)
    return x, y

# Stroke one quadrilateral per observation.
overlay = lunar_map.copy()
for _, row in new_df.iterrows():
    ul_x, ul_y = lat_lon_to_pixel(float(row['V0_lat']), float(row['V0_lon']), width, height)
    ur_x, ur_y = lat_lon_to_pixel(float(row['V1_lat']), float(row['V1_lon']), width, height)
    ll_x, ll_y = lat_lon_to_pixel(float(row['V2_lat']), float(row['V2_lon']), width, height)
    lr_x, lr_y = lat_lon_to_pixel(float(row['V3_lat']), float(row['V3_lon']), width, height)

    # Colour keyed to this observation's share of the chosen element.
    normalized_value = norm(row[RATIO_COLUMN])
    color = scalar_map.to_rgba(row[RATIO_COLUMN], bytes=True)[:3]
    box_color = tuple(int(c) for c in color)
    d_opacity = min_opacity + normalized_value*(max_opacity - min_opacity)

    cv2.line(overlay, (ul_x, ul_y), (ur_x, ur_y), box_color, 2)
    cv2.line(overlay, (ur_x, ur_y), (lr_x, lr_y), box_color, 2)
    cv2.line(overlay, (lr_x, lr_y), (ll_x, ll_y), box_color, 2)
    cv2.line(overlay, (ll_x, ll_y), (ul_x, ul_y), box_color, 2)

# Alpha-composite the overlay back onto the basemap.
blended_image = np.zeros_like(lunar_map)
cv2.addWeighted(overlay, opacity, lunar_map, 1 - opacity, 0, blended_image)
lunar_map = blended_image[:, :, ::-1]

## Saving the figure

In [ ]:
# Compose the figure with a colour bar.
fig, ax = plt.subplots(figsize=(20, 10))
ax.imshow(lunar_map)
ax.axis('off')

# Add the color bar
cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.5])
cb = plt.colorbar(scalar_map, cax=cbar_ax)
cb.set_label(f'{ELEMENT} share of total fitted flux')

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_png_path = os.path.join(OUTPUT_DIR, f"{ELEMENT.lower()}_overlay_op{int(OPACITY*100):02d}.png")
plt.savefig(output_png_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Overlay written to {output_png_path}")